# EADS quickstart

One decision, end to end: signals in, an auditable verdict out.

In [ ]:
%pip install -q git+https://github.com/nirmaljingar/enterprise-ai-decision-systems.git

In [ ]:
from eads.core.pipeline import DecisionPipeline
from eads.core.types import Actor, DecisionRequest
from eads.decision.decision import DecisionEngine
from eads.governance import GovernanceLayer
from eads.synthetic_data import SupplyChainGenerator

request = DecisionRequest(
    request_id="demo",
    goal="decide replenishment order for SKU-1001",
    signals=SupplyChainGenerator(seed=42).generate(3),
    policy_snapshot={"region": "US", "unit_price": 10.0},
    actor=Actor(id="planner-7", roles=("planner",)),
)
record = DecisionPipeline(governance=GovernanceLayer(), decision_engine=DecisionEngine()).run(request)

print("outcome: ", record.verdict.outcome)     # approved | rejected | escalated
print("reason:  ", record.verdict.reason)
print("executed:", record.execution.status)
print("trace:   ", [step["step"] for step in record.trace])

A region is supplied in the policy snapshot because governance fails closed: an action whose
region it cannot establish is rejected rather than waved through.

`outcome` is three-valued. `escalated` is not a softer `approved` — the decision is withheld and
recorded as awaiting a named role.

Next: [the injection demo](eads_killer_demo.ipynb), where the model is compromised on purpose.